# Recover SmolLM2 Shahnameh model
Evaluate and export the preserved 51-step checkpoint from the two-T4 training run.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.56.2', 'accelerate==1.10.1'], check=True)

In [ ]:
from pathlib import Path
script = '"""Evaluate and export the completed Kaggle checkpoint without distributed collectives."""\nimport gc\nimport json\nimport math\nimport shutil\nfrom pathlib import Path\n\nimport torch\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\n\nMODEL_ID = "HuggingFaceTB/SmolLM2-360M"\nWORK = Path(\'/kaggle/working\')\nstates = list(Path(\'/kaggle/input\').rglob(\'trainer_state.json\'))\nassert states, \'Attach the output of the completed SmolLM2 training notebook.\'\nstate_path = max(states, key=lambda p: json.loads(p.read_text())[\'global_step\'])\nstate = json.loads(state_path.read_text())\nassert state[\'global_step\'] == 51 and state[\'epoch\'] == 3.0, state\ncheckpoint = state_path.parent.parent / Path(state[\'best_model_checkpoint\']).name\nassert (checkpoint / \'model.safetensors\').exists(), checkpoint\ndata = next(Path(\'/kaggle/input\').rglob(\'all_val.jsonl\')).parent\ntokenizer = AutoTokenizer.from_pretrained(checkpoint)\nassert tokenizer.chat_template, \'Missing saved training template\'\ndevice = \'cuda:0\' if torch.cuda.is_available() else \'cpu\'\nprint(\'Recovered:\', checkpoint, \'steps:\', state[\'global_step\'], \'device:\', device, flush=True)\n\n\ndef evaluate(model, rows):\n    losses = []\n    for row in rows:\n        messages = row[\'messages\']\n        prompt = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)\n        prefix = tokenizer.encode(prompt, add_special_tokens=False)\n        answer = tokenizer.encode(messages[-1][\'content\'], add_special_tokens=False) + [tokenizer.eos_token_id]\n        ids = torch.tensor([prefix + answer], device=device)\n        labels = torch.tensor([[-100] * len(prefix) + answer], device=device)\n        with torch.inference_mode(), torch.autocast(\'cuda\', dtype=torch.float16, enabled=device.startswith(\'cuda\')):\n            loss = model(input_ids=ids, attention_mask=torch.ones_like(ids), labels=labels).loss.item()\n        assert math.isfinite(loss), loss\n        losses.append(loss)\n    return sum(losses) / len(losses)\n\n\nrows = {lang: [json.loads(line) for line in (data / f\'{lang}_val.jsonl\').read_text().splitlines()]\n        for lang in (\'all\', \'fa\', \'en\', \'de\')}\nbaseline_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32).to(device).eval()\nbaseline = {lang: evaluate(baseline_model, items) for lang, items in rows.items()}\nprint(\'Baseline:\', baseline, flush=True)\ndel baseline_model\ngc.collect()\nif torch.cuda.is_available():\n    torch.cuda.empty_cache()\nmodel = AutoModelForCausalLM.from_pretrained(checkpoint, dtype=torch.float32).to(device).eval()\nassert model.get_input_embeddings().weight.data_ptr() == model.get_output_embeddings().weight.data_ptr()\nfinal = {lang: evaluate(model, items) for lang, items in rows.items()}\nprint(\'Final:\', final, flush=True)\noutput = WORK / \'smollm2-shahnameh-model\'\noutput.mkdir(exist_ok=True)\nmodel.config.use_cache = True\nmodel.save_pretrained(output, safe_serialization=True)\ntokenizer.save_pretrained(output)\nsummary = {\'model\': MODEL_ID, \'training_gpus\': 2, \'train_rows\': 270, \'validation_rows\': 30,\n           \'epochs\': state[\'epoch\'], \'global_step\': state[\'global_step\'],\n           \'best_checkpoint\': checkpoint.name, \'baseline_loss\': baseline, \'final_loss\': final,\n           \'metric\': \'mean of per-example assistant-token cross entropy; FP16 autocast\',\n           \'recovery\': \'Training completed; original DDP job timed out after checkpoint loading. Exported preserved best checkpoint.\',\n           \'history\': state[\'log_history\']}\n(output / \'training_summary.json\').write_text(json.dumps(summary, indent=2), encoding=\'utf-8\')\nsamples = []\nfor lang in (\'fa\', \'en\', \'de\'):\n    row = rows[lang][0]\n    prompt = tokenizer.apply_chat_template(row[\'messages\'][:-1], tokenize=False, add_generation_prompt=True)\n    inputs = tokenizer(prompt, add_special_tokens=False, return_tensors=\'pt\').to(device)\n    with torch.inference_mode(), torch.autocast(\'cuda\', dtype=torch.float16, enabled=device.startswith(\'cuda\')):\n        result = model.generate(**inputs, max_new_tokens=300, do_sample=False, pad_token_id=tokenizer.eos_token_id)\n    answer = tokenizer.decode(result[0, inputs[\'input_ids\'].shape[1]:], skip_special_tokens=True)\n    samples.append({\'language\': lang, \'question\': row[\'messages\'][-2][\'content\'], \'answer\': answer})\n    print(json.dumps(samples[-1], ensure_ascii=False), flush=True)\n(output / \'sample_answers.json\').write_text(json.dumps(samples, ensure_ascii=False, indent=2), encoding=\'utf-8\')\narchive = shutil.make_archive(str(output), \'zip\', output)\nprint(\'EXPORTED:\', archive, Path(archive).stat().st_size, \'bytes\', flush=True)\n'
Path('/kaggle/working/recover_smollm2.py').write_text(script, encoding='utf-8')
subprocess.run([sys.executable, '-u', '/kaggle/working/recover_smollm2.py'], check=True)